In [3]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("data/raw/synthetic_microbiology_dataset")
OUT_DIR = Path("data/processed")
OUT_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", 30)

In [4]:
patients = pd.read_csv(DATA_DIR / "patients.csv")
cultures = pd.read_csv(DATA_DIR / "cultures.csv", parse_dates=["sample_date", "admission_date"])
isolates = pd.read_csv(DATA_DIR / "isolates.csv")
susc = pd.read_csv(DATA_DIR / "susceptibility.csv")

for name, df in [("patients", patients), ("cultures", cultures),
                 ("isolates", isolates), ("susceptibility", susc)]:
    print(f"{name:15} {df.shape[0]:>8,} rows  {df.shape[1]} cols")

patients          10,000 rows  4 cols
cultures          46,792 rows  16 cols
isolates          24,571 rows  9 cols
susceptibility   172,381 rows  8 cols


In [5]:
checks = {
    "cultures with unknown patient": (~cultures.patient_id.isin(patients.patient_id)).sum(),
    "isolates with unknown culture": (~isolates.culture_id.isin(cultures.culture_id)).sum(),
    "susceptibility with unknown isolate": (~susc.isolate_id.isin(isolates.isolate_id)).sum(),
    "n_isolates mismatches": (isolates.groupby("culture_id").size()
                              .reindex(cultures.culture_id, fill_value=0).values
                              != cultures.n_isolates.values).sum(),
    "duplicate isolate+antibiotic": susc.duplicated(["isolate_id", "antibiotic"]).sum(),
}
pd.Series(checks, name="count")

cultures with unknown patient          0
isolates with unknown culture          0
susceptibility with unknown isolate    0
n_isolates mismatches                  0
duplicate isolate+antibiotic           0
Name: count, dtype: int64

In [6]:
def surveillance_year(dates):
    # 12-month periods running October to September
    start = dates.dt.year.where(dates.dt.month >= 10, dates.dt.year - 1)
    return ("Y" + (start - 2022).astype(str) + " ("
            + start.astype(str) + "/" + (start + 1).astype(str).str[-2:] + ")")

cultures["period"] = surveillance_year(cultures.sample_date)
cultures["days_in_hospital"] = (cultures.sample_date - cultures.admission_date).dt.days
cultures["hospital_onset"] = cultures.days_in_hospital >= 2   # sampled on day 3 or later

iso = (isolates
       .merge(cultures, on="culture_id", how="left")
       .merge(patients, on="patient_id", how="left"))

iso.period.value_counts().sort_index()

period
Y1 (2023/24)    8177
Y2 (2024/25)    8333
Y3 (2025/26)    8061
Name: count, dtype: int64

In [7]:
iso = iso.sort_values(["sample_date", "culture_id", "isolate_no"])

first_key = ["patient_id", "organism", "specimen", "period"]
iso["first_isolate"] = ~iso.duplicated(first_key)

n_all, n_first = len(iso), iso.first_isolate.sum()
print(f"All isolates:   {n_all:,}")
print(f"First isolates: {n_first:,} ({n_first / n_all:.1%})")
print(f"Removed:        {n_all - n_first:,}")

All isolates:   24,571
First isolates: 21,053 (85.7%)
Removed:        3,518


In [8]:
INTRINSIC = {
    ("Klebsiella pneumoniae", "AMP"), ("Klebsiella oxytoca", "AMP"),
    ("Citrobacter freundii", "AMP"), ("Proteus mirabilis", "NIT"),
}

iso_cols = ["isolate_id", "patient_id", "period", "sample_date", "specimen", "ward_code",
            "ward_type", "institution", "hospital_onset", "sex", "age_years", "first_isolate"]

ast = susc.drop(columns="culture_id").merge(iso[iso_cols], on="isolate_id", how="left")
ast["resistant"] = (ast.result == "R").astype(int)
ast["intrinsic"] = [(o, a) in INTRINSIC for o, a in zip(ast.organism, ast.antibiotic_code)]

print(f"{ast.first_isolate.sum():,} first-isolate test results")

142,615 first-isolate test results


In [9]:
(ast[~ast.intrinsic]
 .groupby("first_isolate").resistant.mean()
 .rename({True: "first isolates", False: "repeat isolates"})
 .map("{:.1%}".format))

first_isolate
repeat isolates    8.7%
first isolates     8.3%
Name: resistant, dtype: str

In [11]:
iso.to_csv(OUT_DIR / "isolates_joined.csv", index=False)
ast.to_csv(OUT_DIR / "ast_joined.csv", index=False)
print("Saved to", OUT_DIR.resolve())

Saved to /Users/nikolai/Desktop/CultureWatch/data/processed
